In [1]:
import numpy as np
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, Flow, FlowBody, Field

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Assembly creation

### Data 

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:        # DOF prefixes/names (extends default "dof")
  - x             # Recognized as a DOF prefix/name (e.g., "x0" in expressions)
design_names:     # Design parameter prefixes/names (extends default "design")
  - myradius      # Recognized as a parameter prefix/name (e.g., `myradius` in expressions)
  - k
input_names:
  - gravity       # Gravity field

# Default Values (Optional)
defaults:
  myradius0: 1.   # Parameter: Listed in `design_names`
  myradius1: 1.
  k: 1.
  mass: 1.        # Constant: Auto-detected 
  gravity2: -1.

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: myradius0                       
    mass: 1
    position: [-1, 0, 0]             
    force: [k*x + mass*gravity0, mass*gravity1, mass*gravity2]              

  - # Sphere 1 #################
    radius: myradius1                 
    mass: -1
    position: [1+x, 0, 0]       
    force: [-k*x - mass*gravity0, -mass*gravity1, -mass*gravity2]
"""

In [3]:
mybody= SoftBody(yaml_data, verbose=False)

print(repr(mybody))

SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  3 design parameters
  3 input parameters

Default values
  degrees of freedom dof: ['x'] = [ 0.]
  design parameters param: ['k', 'myradius0', 'myradius1'] = [ 1.  1.  1.]
  input parameters param: ['gravity0', 'gravity1', 'gravity2'] = [ 0.  0. -1.]

SPHERE 0
  radius: 1.0
  position: [-1.  0.  0.]
  orientation: [ 0.  0.  0.]
  force: [ 0.  0. -1.]
  torque: [ 0.  0.  0.]

SPHERE 1
  radius: 1.0
  position: [ 1.  0.  0.]
  orientation: [ 0.  0.  0.]
  force: [-0. -0.  1.]
  torque: [ 0.  0.  0.]



## Fluid-structure interaction problem

In [4]:
class PureShearFlow(Flow):
    """Simple pure shear flow u = (y, 0, 0)."""

    def _velocity(self, pos):
        x, y, z = pos  # Extract coordinates
        shear_rate = self.params[0] if self.params else 1  # Default value
        return jnp.array([shear_rate * y, 0, 0])
    
class GravityField(Field):
    """Constant uniform field g = (0, 0, -param[0])."""

    def _field_vector(self, pos, time):
        gravity_acc = self.params[0] if self.params else 1  # Default value
        return jnp.array([0, 0, -gravity_acc])

In [5]:
# Create a shear flow with shear rate 1
shear_flow = PureShearFlow(1)
# Create a gravity field of acceleration 0
gravity_field = GravityField(0)

# Test it (steady case)
pos = [1.0, 2.0, 3.0]  
grad_u = shear_flow.gradient(pos)
Omega, rate_of_strain = shear_flow.omega_s(pos)  
print("Velocity Gradient Tensor (∇u):\n", grad_u)
print("Angular velocity Ω:", Omega)
print("Rate-of-strain E):\n", rate_of_strain)
print("Gravity field", gravity_field.vector(pos))

Velocity Gradient Tensor (∇u):
 [[ 0.  1.  0.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]
Angular velocity Ω: [ 0.   0.  -0.5]
Rate-of-strain E):
 [[ 0.   0.5  0. ]
 [ 0.5  0.   0. ]
 [ 0.   0.   0. ]]
Gravity field [ 0  0  0]


In [39]:
# parameters
final_time = 300 
dt = 0.2
mybody.set_design_defaults(new_dict={'k':3., 'myradius0':0.5})

solver = FlowBody(mybody, shear_flow, gravity_field, dt=dt, init_orientation=[0, 0, 0], integrator="RK2")
trajectory = solver.simulate(T=final_time)

OLD default param values: ['k', 'myradius0', 'myradius1'] [ 3.   0.2  1. ]
NEW default param values: ['k', 'myradius0', 'myradius1'] [ 3.   0.5  1. ]
Time: 0.000 / 300.000  Integrator RK2
Time: 20.000 / 300.000  Integrator RK2
Time: 40.000 / 300.000  Integrator RK2
Time: 60.000 / 300.000  Integrator RK2
Time: 80.000 / 300.000  Integrator RK2
Time: 100.000 / 300.000  Integrator RK2
Time: 120.000 / 300.000  Integrator RK2
Time: 140.000 / 300.000  Integrator RK2
Time: 160.000 / 300.000  Integrator RK2
Time: 180.000 / 300.000  Integrator RK2
Time: 200.000 / 300.000  Integrator RK2
Time: 220.000 / 300.000  Integrator RK2
Time: 240.000 / 300.000  Integrator RK2
Time: 260.000 / 300.000  Integrator RK2
Time: 280.000 / 300.000  Integrator RK2


In [40]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2][0] for t in trajectory])

# Time vector
t = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=t, y=dofs, mode='lines', name='DOF'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()